In [ ]:

print(data.files) # This will list all keys like 'times', 'data', etc.

In [ ]:
import glob
import os
import numpy as np
import os
from astropy.time import Time
from astropy.coordinates import EarthLocation, get_sun
from astropy import units as u
from rafpy.fitting import nonlinear_fit, brute_force_fit, recover_baseline
from rafpy.params import C, FREQ_RF, LAT
from baseline_heatmap import plot_baseline_heatmap
# ==========================================
# 1. OBSERVATION PARAMETERS
# ==========================================
# IMPORTANT: Change these to your dish's exact GPS coordinates!
lo1 = 9.75e9
lo2 = 1.54e9

baseband_center = 125e6
true_sky_freq = lo1 + lo2 + baseband_center
p.lam = C / true_sky_freq
p.lat = 37.871
obs_lon = -122.272 * u.deg   
obs_lat = 37.871 * u.deg     
location = EarthLocation(lat=obs_lat, lon=obs_lon)
p = InterfParams(freq_rf=true_sky_freq)
# ==========================================
# 2. DATA LOADING & SUN TRACKING
# ==========================================
# Assuming files are named 2pm_data_1.npz to 2pm_data_247.npz
import glob
import os

# ==========================================
# 2. DATA LOADING (The Robust Way)
# ==========================================
base_path = "C:/Users/abook/OneDrive/Desktop/UG ASTRO/lab 3 true/"

# This finds ALL .npz files in both folders automatically
#morning_path = os.path.join(base_path, "data 11am", "*.npz")
afternoon_path = os.path.join(base_path, "data_48", "day_data_*.npz")

# Combine the lists of found files
file_list =  glob.glob(afternoon_path) #+glob.glob(morning_path) 

# Sort them so the time-series is in order
import re

def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower() for text in re.split('([0-9]+)', s)]

file_list.sort(key=natural_sort_key)

print(f"Found {len(file_list)} files total.")

all_h_s = []
all_band_real = []
all_declinations = []



def fourier_filter_fringe(data, dt=40, f_lo=0.0005, f_hi=0.01):
    """
    Removes low-frequency drift and high-frequency noise.
    dt: time between integrations in seconds (e.g., 40s)
    f_lo: minimum frequency to keep (Hz)
    f_hi: maximum frequency to keep (Hz)
    """
    n = len(data)
    # Perform the FFT
    freqs = np.fft.fftfreq(n, d=dt)
    fft_vals = np.fft.fft(data)
   
    
    # Create a mask: Keep only frequencies between f_lo and f_hi
    # (Checking both positive and negative frequencies)
    mask = (np.abs(freqs) >= f_lo) & (np.abs(freqs) <= f_hi)
    
    # Apply mask and transform back
    fft_vals[~mask] = 0
    filtered_data = np.fft.ifft(fft_vals)
    
    return np.real(filtered_data)

for f in file_list:
    try:
        with np.load(f) as data:
            spec = data['spec_real']
            jds = np.atleast_1d(data['jd'])

            # Ensure spec is 2D: (time, freq)
            if spec.ndim == 1:
                spec = spec[np.newaxis, :]  # treat as single time step

            for i in range(spec.shape[0]):
                t = Time(jds[i], format='jd', location=location)
                sun_coords = get_sun(t)

                ha = (t.sidereal_time('apparent') - sun_coords.ra).to_value(u.rad)

                all_h_s.append(ha)
                all_band_real.append(spec[i, :])  # full frequency row
                all_declinations.append(sun_coords.dec.to_value(u.rad))

    except Exception as e:
        print(f"Error loading {f}: {e}")

# Stack into 2D array: (total_time_steps, num_frequencies)
h_s_combined = np.array(all_h_s)
band_combined = np.array(all_band_real)
all_declinations = np.array(all_declinations)

valid_mask = ~np.isnan(h_s_combined) & ~np.isnan(band_combined.mean(axis=1))
h_s_clean = h_s_combined[valid_mask]
band_clean = band_combined[valid_mask]
dec_clean = all_declinations[valid_mask]

diffs = np.diff(h_s_clean)
# Filter out any jumps between files/days to get the true integration time
dt_actual = np.median(diffs[diffs > 0]) * (24 * 3600 / (2 * np.pi))
print(f"FIXED Detected dt: {dt_actual:.2f} seconds")
      
if len(all_h_s) == 0:
    print("!!! STILL NO DATA !!!")
    print(f"Checked path: {afternoon_path}")
    # This will help you see if the path itself is wrong
    print(f"Does the folder exist? {os.path.exists(os.path.dirname(afternoon_path))}")
else:
    print(f"Successfully loaded {len(all_h_s)} integrations.")

h_s_combined = np.array(all_h_s)
band_combined = np.array(all_band_real)
mean_delta_rad = np.mean(all_declinations)

band_filtered = fourier_filter_fringe(band_clean, dt=dt_actual, f_lo=0.0005, f_hi=0.1)
# ==========================================
# 3. RUNNING THE FIT
# ==========================================
# p = InterfParams()

print("Running brute force fit...")
bf_result = brute_force_fit(h_s_clean, band_filtered.mean(axis=1), 
                            n_ew=900, b_ew_approx=14, lam=p.lam)

print("Running non-linear fit...")
nl_result = nonlinear_fit(h_s_clean, band_filtered.mean(axis=1), p0_dict=bf_result)
# ==========================================
# 4. EXTRACTING BASELINES & PLOTTING
# ==========================================
# Extract Physical Baselines using the Sun's mean declination
final_lat = obs_lat.to_value(u.rad) if hasattr(obs_lat, 'unit') else np.radians(obs_lat)
final_lam = p.lam.to_value(u.m) if hasattr(p.lam, 'unit') else p.lam
final_dec = mean_delta_rad # should already be float rad

# Re-calculate with clean floats
b_ew, b_ns = recover_baseline(
    nl_result['Q_ew'], 
    nl_result['Q_ns'], 
    final_dec, 
    final_lat, 
    final_lam
)


# Convert uncertainties
sigma_ew = nl_result['sigma_Q_ew'] * p.lam / np.cos(mean_delta_rad)
sigma_ns = nl_result['sigma_Q_ns'] * p.lam / (np.sin(p.lat) * np.cos(mean_delta_rad))

# Extract Correlation
rho = nl_result['cov'][2, 3] / (nl_result['sigma_Q_ew'] * nl_result['sigma_Q_ns'])

print(f"Found Best Fit: b_ew = {b_ew:.2f}m, b_ns = {b_ns:.2f}m")

# Launch the interactive heatmap
plot_baseline_heatmap(
    b_ew=b_ew, 
    b_ns=b_ns, 
    sigma_ew=sigma_ew, 
    sigma_ns=sigma_ns, 
    rho=rho
)

# Check what the keys actually are
print("Keys in nl_result:", nl_result.keys())



In [ ]:
import matplotlib.pyplot as plt
from rafpy.plotting import plot_mf_theory

def main():
    print("Generating the ideal beat pattern fringe modulator graph...")
    
    # Call the built-in plotting function from your package
    # ff_R_max controls how far along the x-axis the plot goes (showing more or fewer beats)
    # N controls the accuracy of the summation in mf_theory
    fig, ax = plot_mf_theory(
        ff_R_max=6.0, 
        N=1000, 
        title='Ideal Beat Pattern Fringe Modulator (Uniform Disk)'
    )
    
    # Save the generated figure to your current working directory
    output_filename = 'ideal_fringe_modulator.png'
    fig.savefig(output_filename, dpi=150)
    print(f"Graph successfully saved as '{output_filename}'")
    
    # Display the plot in an interactive window
    plt.show()

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import os
import matplotlib.pyplot as plt
from rafpy.modulator import mf_theory, mf_theory_zero_crossings
plt.rcParams.update({
    'font.size': 17,          # General text
    'axes.labelsize': 20,     # X and Y labels
    'axes.titlesize': 20
    ,     # Plot title
    'xtick.labelsize': 14,    # X-axis tick numbers
    'ytick.labelsize': 14,    # Y-axis tick numbers
    'legend.fontsize': 16,    # Legend text
    'lines.linewidth': 3.0,   # Thicker lines for visibility
    'axes.linewidth': 2.0,    # Thicker plot border
    'figure.dpi': 150         # Higher resolution for the report
})
def plot_theoretical_fringes_with_null():
    # 1. Generate the theoretical envelope
    # We use ff_R (fringe frequency * radius) as the base x-axis. 
    # Going up to 3.0 gives us a good view of a few beat nodes.
    ff_R_arr = np.linspace(0, 3.0, 2000)
    envelope = mf_theory(ff_R_arr, N=1000)
    
    # 2. Simulate the fast fringe pattern inside the envelope
    # We multiply the envelope by a fast-oscillating cosine.
    # The multiplier '20' is an arbitrary high frequency to mimic the fast 
    # local fringe frequency relative to the slow spatial resolution envelope.
    fast_phase = 2 * np.pi * 20 * ff_R_arr
    fringe_pattern = envelope * np.cos(fast_phase)
    
    # 3. Find the exact first null crossing programmatically
    # mf_theory_zero_crossings returns the roots in units of ff*R
    zeros = mf_theory_zero_crossings(n_zeros=1, N=1000)
    first_null_ff_R = zeros[0]
    
    # In your plotting.py, the x-axis is 'Source Diameter * f_f', which is 2 * R * f_f.
    # So we multiply our ff_R arrays by 2 to match your lab's convention.
    x_axis_plot = 2 * ff_R_arr
    first_null_x = 2 * first_null_ff_R
    
    # 4. Plotting
    plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True, 'grid.alpha': 0.3})
    fig, ax = plt.subplots(figsize=(8, 7))
    
    # Plot the fast fringe oscillations
    ax.plot(x_axis_plot, fringe_pattern, color='steelblue', alpha=0.8, 
            label='Simulated Fringe Oscillations')
            
    # Plot the upper and lower bounds of the envelope
    ax.plot(x_axis_plot, envelope, 'r--', lw=1.5, label='Envelope Modulator (+MF)')
    ax.plot(x_axis_plot, -envelope, 'r--', lw=1.5, label='Envelope Modulator (-MF)')
    
    # Highlight the first null
    ax.plot(first_null_x, 0, 'go', ms=8, zorder=5, 
            label=f'First Null (x ≈ {first_null_x:.3f})')
    ax.axvline(first_null_x, color='green', linestyle=':', alpha=0.8)
    
    # Formatting
    ax.axhline(0, color='k', lw=0.8)
    ax.set_xlabel('Source Diameter × f_f  (= 2R × f_f)')
    ax.set_xlim(0,4)
    ax.set_ylabel('Fringe Amplitude')
    ax.set_title('Theoretical Fringe Pattern within Modulator Envelope')
    ax.legend(loc='upper right')
    
    fig.tight_layout()
    
    # Save and display
    output_filename = 'fringe_pattern_with_null.png'
    output_dir = "C:/Users/abook/OneDrive/Desktop/UG ASTRO/lab 3 true/graphs"
    fig.savefig(os.path.join(output_dir, output_filename), dpi=300)
    print(f"Graph saved! First null occurs at Diameter × f_f ≈ {first_null_x:.3f}")
    plt.show()

if __name__ == "__main__":
    plot_theoretical_fringes_with_null()

In [ ]:
import glob
import os
import numpy as np
from astropy.time import Time
from astropy.coordinates import EarthLocation, get_sun
from astropy import units as u
from rafpy.plotting import plot_waterfall
import matplotlib.pyplot as plt
# ==========================================
# 1. SETUP PATHS & LOCATION
# ==========================================
base_path = "C:/Users/abook/OneDrive/Desktop/UG ASTRO/lab 3 true/"
afternoon_path = os.path.join(base_path, "data_48", "day_data_*.npz")

obs_lon = -122.272 * u.deg   
obs_lat = 37.871 * u.deg     
location = EarthLocation(lat=obs_lat, lon=obs_lon)

file_list_complex = glob.glob(afternoon_path)
file_list_complex.sort()

print(f"Found {len(file_list_complex)} files for complex import.")

# ==========================================
# 2. LOAD COMPLEX DATA
# ==========================================
all_h_s_complex = []
all_times_unix = []
all_complex_vis = []

for f in file_list_complex:
    try:
        with np.load(f) as data:
            # ---> IMPORTANT: Check these keys match what you actually saved! <---
            spec_real = data['spec_real']
            
            if 'spec_imag' in data:
                spec_imag = data['spec_imag']
            else:
                # If you get this print statement, you need to check your npz keys!
                print(f"Keys in {os.path.basename(f)}: {data.files}")
                raise KeyError("Could not find imaginary data key in the npz file.")

            # Combine into a complex array (Real + j*Imag)
            spec_complex = spec_real + 1j * spec_imag
            
            jds = np.atleast_1d(data['jd'])

            # Ensure spec is 2D: (time, freq)
            if spec_complex.ndim == 1:
                spec_complex = spec_complex[np.newaxis, :]  

            for i in range(spec_complex.shape[0]):
                t = Time(jds[i], format='jd', location=location)
                sun_coords = get_sun(t)

                ha = (t.sidereal_time('apparent') - sun_coords.ra).to_value(u.rad)

                all_h_s_complex.append(ha)
                all_times_unix.append(t.unix)
                all_complex_vis.append(spec_complex[i, :])  

    except Exception as e:
        print(f"Error loading {f}: {e}")

# Stack into 2D arrays: (total_time_steps, num_frequencies)
all_complex_vis = np.array(all_complex_vis)  # shape: (N, F)
all_times_unix = np.array(all_times_unix)
all_h_s_complex = np.array(all_h_s_complex)

sort_indices = np.argsort(all_times_unix)

all_times_unix = all_times_unix[sort_indices]
all_h_s_complex = all_h_s_complex[sort_indices]
all_complex_vis = all_complex_vis[sort_indices, :]

if len(all_complex_vis) > 0:
    print(f"Successfully loaded {all_complex_vis.shape[0]} time steps and {all_complex_vis.shape[1]} frequency channels.")

    # ==========================================
    # 3. CONSTRUCT RAFPY DICTIONARY & PLOT
    # ==========================================
    
    # Define your frequency axis (Make sure this matches your actual hardware!)
    # Example: 2048 channels across your baseband bandwidth (e.g., 250 MHz or 500 MHz)
    num_channels = all_complex_vis.shape[1]
    
    # UPDATE THIS to your actual IF or RF frequency array
    # If your baseband is 250 MHz: np.linspace(0, 250e6, num_channels)
    frequencies_hz = np.linspace(0, 250e6, num_channels) 

    # Build the dictionary rafpy expects
    vis_dict = {
        'times': all_times_unix,
        'chan_freqs': frequencies_hz,
        'vis_cube': all_complex_vis,
        'amp': np.abs(all_complex_vis),
        'phase': np.angle(all_complex_vis)
    }

    print("Generating Phase Waterfall Plot...")
    fig, ax = plot_waterfall(vis_dict, component='phase', title='Visibility Phase Waterfall')
    fig.set_size_inches(10, 8)
    
    # Save the waterfall plot
    fig.savefig('phase_waterfall_plot.png', dpi=200, bbox_inches='tight')
    print("Waterfall plot saved as 'phase_waterfall_plot.png'")

    # ==========================================
    # 4. CALCULATE GEOMETRIC DELAY (tau_g)
    # ==========================================
    # Pick a snapshot in the middle of the observation
    mid_idx = len(all_times_unix) // 2

    # Unwrap the phase for this specific time step
    phase_rad = np.unwrap(vis_dict['phase'][mid_idx, :]) 

    # Fit a linear line (degree 1) to Phase vs Frequency
    slope, intercept = np.polyfit(frequencies_hz, phase_rad, 1)

    # slope = d(phi)/d(nu) = 2 * pi * tau_g
    tau_g_measured = slope / (2 * np.pi)

    print(f"\n--- Delay Calculation ---")
    print(f"Snapshot Time (HA): {all_h_s_complex[mid_idx]:.4f} rad")
    print(f"Measured phase slope: {slope:.3e} rad/Hz")
    print(f"Geometric Delay (tau_g): {tau_g_measured * 1e9:.3f} nanoseconds")

else:
    print("No complex data was loaded.")

f_min = 80e6  
f_max = 170e6  

# Create a mask to slice the frequency and visibility arrays
f_mask = (frequencies_hz >= f_min) & (frequencies_hz <= f_max)

freqs_zoomed = frequencies_hz[f_mask]
vis_zoomed = all_complex_vis[:, f_mask]

# Calculate time in minutes for the Y-axis
t_minutes = (all_times_unix - all_times_unix[0]) / 60.0

# 2. Create the custom plot
fig, ax = plt.subplots(figsize=(10, 8))

# Switching to 'hsv' for that classic, seamless phase wrap rainbow
im = ax.pcolormesh(
    freqs_zoomed / 1e6,        # X-axis in MHz
    t_minutes,                 # Y-axis in Minutes
    np.angle(vis_zoomed),      # Phase data in radians
    cmap='twilight_shifted',                # <--- Changed colormap
    shading='auto'
)

# 3. Automate the Y-axis limits
# This automatically sets the axis to the exact length of your data
ax.set_ylim(t_minutes.max(), t_minutes.min()) 

# Formatting
ax.set_xlabel('Frequency (MHz)', fontsize=14)
ax.set_ylabel('Time since observation start (minutes)', fontsize=18)
ax.set_title('Visibility Phase Waterfall', fontsize=18)
fig.colorbar(im, ax=ax, label='Phase (Radians)')

plt.tight_layout()
fig.savefig('zoomed_hsv_waterfall.png', dpi=300)
plt.show()

In [ ]:
# from astropy.time import Time
from astropy.coordinates import get_sun
from rafpy.fringe import geometric_delay
from rafpy.params import LAT

# ==========================================
# 5. CALCULATE INSTRUMENTAL CABLE DELAY (tau_c)
# ==========================================

# 1. Grab the exact time and hour angle from our snapshot
t_mid = Time(all_times_unix[mid_idx], format='unix')
ha_mid = all_h_s_complex[mid_idx]

# 2. Get the Sun's declination at this exact moment
delta_mid = get_sun(t_mid).dec.to_value(u.rad)

# 3. Calculate Theoretical Geometric Delay
# ---> UPDATE THESE WITH YOUR FITTED BASELINES <---
b_ew_fitted = 20.0  # e.g., 19.85
b_ns_fitted = 0.0   # e.g., 1.12

# Your rafpy package calculates the theoretical tau_g in seconds
tau_g_theory = geometric_delay(
    h_s=ha_mid, 
    delta=delta_mid, 
    b_ew=b_ew_fitted, 
    b_ns=b_ns_fitted, 
    lat=LAT
)

# 4. Calculate Cable Delay
tau_c = tau_g_measured - tau_g_theory

print(f"\n--- Cable Delay Calculation at HA = {ha_mid:.4f} rad ---")
print(f"1. Measured Total Delay:      {tau_g_measured * 1e9:8.3f} ns")
print(f"2. Theoretical Geo Delay:     {tau_g_theory * 1e9:8.3f} ns")
print(f"3. Instrumental Cable Delay:  {tau_c * 1e9:8.3f} ns")

# Optional: Convert nanoseconds of delay into a physical cable length difference
# (Assuming the signal travels through the coax cables at ~0.8c)
speed_of_light = 299792458
velocity_factor = 0.8  
cable_length_diff = tau_c * (speed_of_light * velocity_factor)

print(f"\n-> This means one cable is effectively ~{abs(cable_length_diff):.2f} meters longer than the other!")

In [ ]:
print(f"DEBUGGING PARAMETERS:")
print(f"1. Wavelength (lam):  {lam} (Should be ~0.03 if in meters)")
print(f"2. East-West (b_ew):  {b_ew_fit} (Check units!)")
print(f"3. Max B_proj/lambda: {np.max(b_proj_lambda):.2f} wavelengths")
print(f"4. Max Bessel Arg x:  {np.max(x_args):.2f} (Needs to be > 3.83 to see a null!)")

In [ ]:
# ==========================================
# 1. SETUP PARAMETERS
# ==========================================
b_ew_fit = b_ew  
b_ns_fit = b_ns  
lat_rad = obs_lat.to_value(u.rad)
dec_rad = mean_delta_rad

# ---> FIXING THE WAVELENGTH <---
# Assuming an X-Band Frequency (e.g., 10.4 GHz). 
# Replace 10.4e9 with your actual telescope frequency!
speed_of_light = 299792458  # meters/second
observing_frequency_hz = 10.4e9  # 10.4 GHz

# Overwrite the broken p.lam
lam = C / (lo1 + lo2 + baseband_center)   # 0.02881 m — consistent with your LO settings

# Print to verify
print(f"Corrected Wavelength: {lam:.4f} meters")

solar_diameter_arcmin = 34.0

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from rafpy.modulator import mf_observed, fit_diameter, mf_theory
from rafpy.fringe import local_fringe_freq

# ==========================================
# 1. CALCULATE SPATIAL FREQUENCY & ENVELOPE
# ==========================================

# Calculate the absolute spatial frequency (cycles per radian)
# This maps our time-series (h_s) to the physical baseline projection
ff_rad = local_fringe_freq(h_s_clean, mean_delta_rad, b_ew, b_ns, p.lat, p.lam, in_hz=False)
ff_abs = np.abs(ff_rad)

# Use rafpy to bin the data and extract the observed envelope (Visibility)
# We use the mean of the filtered band
mf_obs_result = mf_observed(
    h_s_clean, 
    band_filtered.mean(axis=1), 
    nl_result, 
    delta=mean_delta_rad, 
    b_ew=b_ew, 
    b_ns=b_ns, 
    lat=p.lat, 
    lam=p.lam, 
    n_bins=50
)

# Fit the diameter based on the binned envelope
diam_result = fit_diameter(mf_obs_result)
best_R = diam_result['R_rad']
best_D = diam_result['diameter_deg']

print(f"--- Diameter Fit Result ---")
print(f"Measured Sun Diameter: {best_D:.4f}°")

# ==========================================
# 2. CREATE THE ENVELOPE PLOT
# ==========================================
plt.figure(figsize=(12, 7), dpi=150)

# A. Plot the Raw Fringes (Blue)
# We plot the filtered real signal vs absolute spatial frequency
plt.plot(ff_abs, band_filtered.mean(axis=1), color='royalblue', alpha=0.3, lw=0.5, label='Fringe Pattern')

# B. Plot the Binned Visibility Points (Black Dots)
# These represent the 'amplitude' of the fringes in different frequency bins
plt.errorbar(mf_obs_result['ff_rad_bins'], mf_obs_result['mf_obs'], 
             yerr=mf_obs_result['mf_err'], fmt='o', color='black', 
             markersize=4, label='Binned Visibility (MF_obs)', capsize=2, zorder=5)

# C. Plot the Theoretical Bessel Envelope (Red Dashed)
# We generate a smooth curve using the rafpy integration model
ff_plot = np.linspace(0, np.max(ff_abs), 1000)
# Amplitude A comes from your nonlinear fit result
envelope_curve = nl_result['A'] * mf_theory(ff_plot * best_R)

plt.plot(ff_plot, envelope_curve, color='firebrick', lw=2.5, linestyle='--', 
         label=f'Bessel Fit (D = {best_D:.3f}°)')
plt.plot(ff_plot, -envelope_curve, color='firebrick', lw=2.5, linestyle='--') # Lower bound

# ==========================================
# 3. FORMATTING
# ==========================================
plt.title("Solar Visibility: High-Frequency Fringes & Bessel Envelope", fontsize=16)
plt.xlabel("Spatial Fringe Frequency $f_f$ [cycles/radian]", fontsize=14)
plt.ylabel("Visibility Amplitude (Filtered)", fontsize=14)

# Typical X-axis for a 20m baseline at X-band is 0 to ~600 cycles/rad
plt.xlim(0, np.max(ff_abs) * 1.05)
# Set Y-axis to frame the envelope nicely
plt.ylim(-nl_result['A']*1.4, nl_result['A']*1.4)

plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='upper right', frameon=True, shadow=True)

plt.tight_layout()
plt.savefig("sun_visibility_envelope.png")
plt.show()